# MNIST autoencoder — analyze any wandb run

Set `RUN_ID` to any run trained by `train.py`. The model is rebuilt from the run's
logged config and its checkpoint (local copy if present, else the wandb artifact).

In [ ]:
%load_ext autoreload
%autoreload 2
from pathlib import Path

import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
import wandb

from chimera.models import ConvAutoEncoder
from chimera.data import MNISTDataModule
from train import MODEL_CONFIG, IMAGE_SIZE, PROJECT_DEFAULT, OUTPUTS

RUN_ID = "PASTE_RUN_ID_HERE"
PROJECT = PROJECT_DEFAULT
device = "cuda" if torch.cuda.is_available() else "cpu"

In [ ]:
# Pull the run's config + locate its checkpoint (local first, else wandb artifact).
def load_run(run_id, project):
    model_cfg, ckpt = MODEL_CONFIG, OUTPUTS / run_id / "last.ckpt"
    try:
        api = wandb.Api()
        run = api.run(f"{api.default_entity}/{project}/{run_id}")
        model_cfg = dict(run.config)["model"]
        if not ckpt.exists():
            for art in reversed(list(run.logged_artifacts())):
                if art.type == "model":
                    ckpt = sorted(Path(art.download()).glob("*.ckpt"))[0]
                    break
    except (wandb.errors.CommError, wandb.errors.UsageError) as e:
        # Only fall back for connectivity/auth issues. A bad RUN_ID, missing
        # config["model"], or missing artifact is a real error and should surface.
        print(f"wandb API unavailable ({e}); using local ckpt + MODEL_CONFIG")
    return model_cfg, Path(ckpt)


model_cfg, ckpt_path = load_run(RUN_ID, PROJECT)
print("checkpoint:", ckpt_path)
assert ckpt_path.exists(), f"checkpoint not found: {ckpt_path}"

model = ConvAutoEncoder(**model_cfg)
state = torch.load(ckpt_path, map_location="cpu", weights_only=False)["state_dict"]
state = {
    k.removeprefix("model."): v for k, v in state.items() if k.startswith("model.")
}
model.load_state_dict(state)
model = model.to(device).eval()
print("loaded", sum(p.numel() for p in model.parameters()), "params")

In [ ]:
# Test data, materialized to 32x32 the same way training does (no augmentation).
dm = MNISTDataModule(batch_size=512, num_workers=0, image_size=IMAGE_SIZE)
dm.prepare_data()
dm.setup("test")


def prep(images):
    # The DataModule already yields IMAGE_SIZE bf16 batches; just cast + move.
    return images.float().to(device)

In [ ]:
# Reconstruction grid: top row inputs, bottom row reconstructions.
images, _ = next(iter(dm.test_dataloader()))
x = prep(images[:8])
with torch.inference_mode():
    recon = model(x).float().clamp(0, 1).cpu()
x = x.float().cpu()

fig, axes = plt.subplots(2, 8, figsize=(12, 3))
for i in range(8):
    axes[0, i].imshow(x[i, 0], cmap="gray")
    axes[0, i].axis("off")
    axes[1, i].imshow(recon[i, 0], cmap="gray")
    axes[1, i].axis("off")
axes[0, 0].set_ylabel("input")
axes[1, 0].set_ylabel("recon")
plt.tight_layout()
plt.show()

In [ ]:
# Latent scatter: encode -> flatten 4x2x2=16-dim -> 2D via torch.pca_lowrank.
zs, ys = [], []
with torch.inference_mode():
    for images, labels in dm.test_dataloader():
        z = model.encode(prep(images)).flatten(1).float().cpu()
        zs.append(z)
        ys.append(labels)
z = torch.cat(zs)
y = torch.cat(ys)

z = z - z.mean(0)
_, _, V = torch.pca_lowrank(z, q=2)
proj = (z @ V[:, :2]).numpy()

plt.figure(figsize=(7, 6))
sc = plt.scatter(proj[:, 0], proj[:, 1], c=y.numpy(), cmap="tab10", s=4, alpha=0.5)
plt.colorbar(sc, label="digit")
plt.title(f"latent PCA — run {RUN_ID}")
plt.show()